# CSC 786 Phase 2 — Robustness and Deployment Efficiency

This notebook records the completed sampled experiments on public NetFlow v2 releases of CSE-CIC-IDS2018 and UNSW-NB15, plus an optional full-source UNSW experiment. It reuses the implemented source-only preprocessing and validation-threshold protocol.

The main comparison uses 250,000 uniformly sampled flows per dataset, three seeds (13, 29, 47), three source training fractions (25%, 50%, 100%), and both transfer directions. Read `data/source_manifest.json` for provenance, checksums, and citation. Raw full source data are kept outside the ZIP. Feature-name alignment does not guarantee semantic or unit equivalence, so findings remain exploratory.

Target labels are reserved for final evaluation. The results section reads the saved CSVs; rerunning the training cells regenerates the experiments. Dynamic INT8 outputs collapsed in these runs, so they are a diagnostic failure rather than a deployable compressed model.

## 1. Environment and reproducibility
Install `requirements.txt` if needed. XGBoost, LightGBM, and ONNX dependencies are optional for the corresponding experiments.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.phase2 import run_pair
config = json.loads((ROOT / "configs/phase2.json").read_text())
config


## 2. Inspect prepared files and label mapping
Check each CSV's columns, label values, category mapping, and data provenance. Shared column names alone do not prove equal feature definitions or units.

In [ ]:
for pair in config["pairs"]:
    for key in ("source_csv", "target_csv"):
        path = Path(pair[key])
        print(path, "exists:", path.exists())
        if path.exists():
            df = pd.read_csv(path, nrows=5)
            print(df.columns.tolist())
            print(df.head(2))


## 3. Repeated cross-dataset experiments
Each direction uses stratified 70/15/15 source train/validation/test partitions for each seed. Exact duplicate feature vectors are assigned to a single split group rather than removed. Preprocessing fits on the source training partition, thresholds come from source validation, and target labels are used only for final metrics. Sample-size runs subsample source training data only.

In [ ]:
runs = []
for pair in config["pairs"]:
    result = run_pair(pair["source_csv"], pair["target_csv"], pair["source_name"],
                      pair["target_name"], config, ROOT / config.get("output_dir", "results"))
    runs.append(result)
    print(pair["source_name"], "→", pair["target_name"], result[-1])


## 4. Results, repeats, and uncertainty
Saved result tables contain per-seed metrics and mean, standard deviation, and 95% Student-t intervals across three seeds. These intervals describe split/seed variability, not independent population sampling. The full-development logistic experiments use one seed and therefore do not have a repeated-seed interval.

In [ ]:
result_files = sorted((ROOT / "results/tables").glob("model_results_*.csv"))
for path in result_files:
    print("\n", path.name)
    display(pd.read_csv(path).groupby(["model", "train_fraction", "split"])[
        ["pr_auc", "recall", "precision", "f1", "fpr", "fit_seconds", "inference_rows_per_second"]
    ].agg(["mean", "std", "count"]).round(4))


## 5. Distribution shift, ablation, and categories
Interpret feature tests, domain classifier AUC, feature importance, ablation, and attack-category scores as diagnostics. A shifted feature is not a causal explanation. Small categories need uncertainty intervals and may not support stable ranking.

In [ ]:
for path in sorted((ROOT / "results/tables").glob("shift_*.csv")):
    shift = pd.read_csv(path).sort_values("wasserstein", ascending=False)
    display(shift.head(15))
    ax = shift.head(15).sort_values("wasserstein").plot.barh(x="feature", y="wasserstein",
        figsize=(8, 5), legend=False, title=path.stem.replace("_", " "))
    ax.set_xlabel("Wasserstein distance (feature units)")
    plt.tight_layout(); plt.show()


## 6. Deployment and compression measurements
CPU timings include the scikit-learn pipeline inference path for batch measurements. Report serialized model size and observed process RSS snapshots as environment-specific measurements, not isolated peak memory. Dynamic INT8 ONNX models were smaller but returned nearly constant scores and chance ROC-AUC on target data. Do not use their throughput as a valid deployment comparison.

## 7. Integrity and claim boundaries
- [x] Public dataset versions, source counts, and checksums recorded.
- [x] Shared numeric feature manifest is recorded; semantic/unit harmonization remains a limitation.
- [x] Split seeds and source-only threshold protocol preserved.
- [x] Target labels used only for final evaluation.
- [x] Repeated-seed intervals reported for the primary sample experiments.
- [ ] Independent confirmation on a third dataset or later capture remains future work.
- [ ] Full CSE-CIC training is beyond current compute feasibility.

## 8. Empirical figures
Figures are generated from saved measured outputs. They show cross-domain metrics, uncertainty, feature shifts, domain importance, category recall, feature ablation, and deployment tradeoffs. Three-seed intervals use Student-t critical values.

In [ ]:
from src.figures import make_figures
generated_figures = make_figures(ROOT / "results")
generated_figures


The separate experiment used the entire UNSW v2 source release. It held 359,277 source rows out for final source testing, calibrated the threshold (0.802982) from five-fold grouped out-of-fold predictions on 2,030,998 source-development rows, and refit on all development data. On the 250,000-row CIC target sample, PR-AUC was 0.2286, recall 0.6968, precision 0.2024, F1 0.3137, and FPR 0.3710. The source-test PR-AUC was 0.8440. This is a single-seed supplementary experiment, not independent confirmation. To reproduce, run `python src/full_source_unsw.py` from the project root.

In [ ]:
import subprocess, sys
full_source_script = ROOT / "src/full_source_unsw.py"
subprocess.run([sys.executable, str(full_source_script)], cwd=ROOT, check=True)

## 10. Saved tables and run metadata
Use these generated tables as the source of truth. The cross-dataset PR-AUC must be compared with each target dataset’s attack prevalence. Category recall at a frozen source threshold is exploratory, especially where support is small.

In [ ]:
summary = pd.read_csv(ROOT / "results/tables/phase2_summary_mean_std.csv")
display(summary[summary.split == "cross_dataset"].sort_values(["direction", "model", "train_fraction"]))
for path in sorted((ROOT / "results/tables").glob("full_source_*.csv")):
    display(pd.read_csv(path))
for path in sorted((ROOT / "results/tables").glob("onnx_int8_*.csv")):
    display(pd.read_csv(path))
for path in sorted((ROOT / "results/tables").glob("onnx_*sensitivity.csv")):
    display(pd.read_csv(path))
if (ROOT / "results/tables/onnx_deployment_memory.csv").exists():
    display(pd.read_csv(ROOT / "results/tables/onnx_deployment_memory.csv"))
